In [3]:
# Notebook: nb_silver_sanctions_clean
# Purpose: Clean OpenSanctions entities and prepare for matching
from pyspark.sql import functions as F
from datetime import datetime

print(" Starting Silver Layer OpenSanctions Cleaning...")
print("=" * 60)

# 1. Read from Bronze
bronze_df = spark.table("bronze_sanctions_entities")
print(f" Read {bronze_df.count()} rows from Bronze.")


StatementMeta(, 26cf4aa2-5c32-4696-a1d6-4daedaeafeb4, 5, Finished, Available, Finished, False)

 Starting Silver Layer OpenSanctions Cleaning...
 Read 1000 rows from Bronze.


In [4]:
# 2. Select and rename key fields
# Note: Our synthetic Bronze data doesn't have an 'aliases' column, 
# so we will use the 'name' column as the alias_name to keep the schema consistent for Gold.
sanctions_df = bronze_df.select(
    F.col("id").alias("entity_id"),
    F.col("name").alias("entity_name"),
    F.col("schema").alias("entity_type"),
    F.col("country").alias("country_code"),
    F.col("topics").alias("sanction_topics"),
    F.col("datasets").alias("source_datasets")
)

# 3. Create alias_name 
# (In production with real data, this would come from exploding an aliases array)
# For our synthetic data, the name IS the alias.
exploded_df = sanctions_df.withColumn("alias_name", F.col("entity_name"))

# 4. Clean and standardize
clean_df = exploded_df.withColumn("entity_name", F.upper(F.trim(F.col("entity_name")))) \
                      .withColumn("alias_name", F.upper(F.trim(F.col("alias_name")))) \
                      .withColumn("entity_type", F.upper(F.trim(F.col("entity_type")))) \
                      .withColumn("country_code", F.upper(F.trim(F.col("country_code"))))

# 5. Drop duplicates (same entity + alias combination)
initial_count = clean_df.count()
clean_df = clean_df.dropDuplicates(["entity_id", "alias_name"])
print(f" Removed {initial_count - clean_df.count()} duplicate rows.")


StatementMeta(, 26cf4aa2-5c32-4696-a1d6-4daedaeafeb4, 6, Finished, Available, Finished, False)

 Removed 500 duplicate rows.


In [5]:
# 6. Add Data Quality & Load Timestamp
clean_df = clean_df.withColumn("dq_score", F.lit(100)) \
                   .withColumn("silver_load_timestamp", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))

# 7. Write to Silver Delta Table
clean_df.write.format("delta").mode("overwrite").saveAsTable("silver_sanctions")

print(f"\n Silver Sanctions Cleaning Complete!")
print(f" Total records created: {clean_df.count()}")
print("Table created: silver_sanctions")

# 8. Display sample
print("\n Sample of cleaned sanctions data:")
display(clean_df.select("entity_id", "entity_name", "alias_name", "entity_type", "country_code").limit(5))

StatementMeta(, 26cf4aa2-5c32-4696-a1d6-4daedaeafeb4, 7, Finished, Available, Finished, False)


 Silver Sanctions Cleaning Complete!
 Total records created: 500
Table created: silver_sanctions

 Sample of cleaned sanctions data:


SynapseWidget(Synapse.DataFrame, d0deecb6-6881-4ae8-a698-3885c5705caa)